# Annotation Quality Evaluation

Analyses the quality of the LLM-produced annotation stored in
`annotation/eval_annotated.parquet`.  The dataset is the **new sentence-based
format** (R26): one row per sentence, each with a `concepts` column that is a
list of dicts `{concept_text, gold_id, candidate_ids}`.

The annotation notebook (`annotate_eval.ipynb`) ran on the cluster and produced
`gold_id` selections (and optionally a `confidence` / `annotation_note` field)
for every concept.  This notebook checks:

1. **Completeness** — how many concepts were actually annotated vs skipped/null
2. **Gold-in-candidates rate** — how often the chosen `gold_id` is among the
   10 candidate IDs that were offered to the model
3. **Candidate diversity** — how many distinct pictogram IDs appear as candidates
4. **Concept-text quality** — length distribution, stop-word ratio, degenerate
   concepts (single char, numeric-only, …)
5. **Gold-ID distribution** — most/least frequent gold IDs, ID reuse across
   concepts
6. **Concepts-per-sentence distribution** — how many concepts each sentence has
7. **Annotation log analysis** — inter-row timing, error rate, batch stats
8. **Sample inspection** — random rows for manual spot-check

## 1. Setup

In [ ]:
import sys
import re
import json
import ast
import math
import random
from collections import Counter
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

import nltk
nltk.download('wordnet',  quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4',  quiet=True)
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

NOTEBOOK_DIR = Path().resolve()
# notebook lives at annotation/ or at project root depending on how it is run
ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'annotation').is_dir() else NOTEBOOK_DIR.parent
ANN_DIR = ROOT / 'annotation'

print(f'ROOT    : {ROOT}')
print(f'ANN_DIR : {ANN_DIR}')

## 2. Dataset load & flatten

In [ ]:
ANNOTATED_PATH  = ANN_DIR / 'eval_annotated.parquet'
FILTERED_PATH   = ANN_DIR / 'eval_filtered.parquet'   # pre-annotation baseline
LOG_PATH        = ANN_DIR / 'annotation_log.jsonl'

df_ann = pd.read_parquet(ANNOTATED_PATH)
df_flt = pd.read_parquet(FILTERED_PATH)

print(f'Annotated  : {len(df_ann):,} rows  cols={list(df_ann.columns)}')
print(f'Filtered   : {len(df_flt):,} rows  cols={list(df_flt.columns)}')
df_ann.head(3)

In [ ]:
# ── Deserialise `concepts` if stored as string ────────────────────────────────
for df in [df_ann, df_flt]:
    if df['concepts'].dtype == object and isinstance(df['concepts'].iloc[0], str):
        df['concepts'] = df['concepts'].apply(ast.literal_eval)

# ── Flatten to concept-level dataframe ───────────────────────────────────────
# Each row in df_concepts = one (sentence, concept) pair
rows = []
for sent_idx, row in df_ann.iterrows():
    for pos, c in enumerate(row['concepts']):
        rows.append({
            'sent_idx':      sent_idx,
            'sentence':      row['sentence'],
            'turn_pos':      pos,
            'n_concepts':    len(row['concepts']),
            'concept_text':  c.get('concept_text', ''),
            'gold_id':       c.get('gold_id'),
            'candidate_ids': c.get('candidate_ids', []),
        })

df = pd.DataFrame(rows)
df['gold_id'] = pd.to_numeric(df['gold_id'], errors='coerce')
df['n_candidates'] = df['candidate_ids'].apply(len)
df['gold_in_candidates'] = df.apply(
    lambda r: (not pd.isna(r['gold_id'])) and (int(r['gold_id']) in r['candidate_ids']),
    axis=1,
)

TOTAL_SENTENCES = len(df_ann)
TOTAL_CONCEPTS  = len(df)
print(f'Flattened: {TOTAL_CONCEPTS:,} concepts from {TOTAL_SENTENCES:,} sentences')
df.head(5)

## 3. Completeness

In [ ]:
n_annotated  = df['gold_id'].notna().sum()
n_null       = df['gold_id'].isna().sum()
pct_complete = n_annotated / TOTAL_CONCEPTS * 100

print(f'Total concepts     : {TOTAL_CONCEPTS:,}')
print(f'Annotated (non-null gold_id): {n_annotated:,}  ({pct_complete:.1f}%)')
print(f'Null / skipped     : {n_null:,}')

# Sentences with at least one null concept
null_sents = df[df['gold_id'].isna()]['sent_idx'].nunique()
print(f'Sentences with ≥1 null concept: {null_sents:,}  ({null_sents/TOTAL_SENTENCES*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Pie: annotated vs null
ax = axes[0]
ax.pie(
    [n_annotated, n_null],
    labels=[f'annotated\n{n_annotated:,}', f'null\n{n_null:,}'],
    autopct='%1.1f%%',
    colors=['#4C9BE8', '#F28B30'],
    startangle=90,
)
ax.set_title('Concept annotation completeness')

# Histogram: null concepts per sentence
ax2 = axes[1]
null_per_sent = df[df['gold_id'].isna()].groupby('sent_idx').size()
if len(null_per_sent):
    ax2.hist(null_per_sent, bins=range(1, null_per_sent.max() + 2), color='#F28B30', edgecolor='white')
    ax2.set_xlabel('null concepts in sentence')
    ax2.set_ylabel('sentences')
    ax2.set_title('Distribution of missing concepts per sentence')
    ax2.spines[['top', 'right']].set_visible(False)
else:
    ax2.text(0.5, 0.5, 'No null concepts', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('No missing concepts')

plt.tight_layout()
plt.show()

## 4. Gold-in-candidates rate

The annotator model was shown 10 candidate IDs and asked to pick the best one.
If `gold_id ∉ candidate_ids` the selection is either hallucinated or the model
picked an ID outside the offered window — a clear annotation error.

In [ ]:
df_a = df[df['gold_id'].notna()].copy()

n_in  = df_a['gold_in_candidates'].sum()
n_out = (~df_a['gold_in_candidates']).sum()
pct   = n_in / len(df_a) * 100

print(f'Gold ID within candidates : {n_in:,}  ({pct:.1f}%)')
print(f'Gold ID OUTSIDE candidates: {n_out:,}  ({100-pct:.1f}%)')

# Show worst offenders
df_out = df_a[~df_a['gold_in_candidates']][['sentence', 'concept_text', 'gold_id', 'candidate_ids']]
print(f'\nSample of hallucinated selections (gold not in candidates):')
display(df_out.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['gold in candidates', 'gold OUTSIDE candidates'],
       [n_in, n_out], color=['#4C9BE8', '#E84C4C'], edgecolor='white')
for i, v in enumerate([n_in, n_out]):
    ax.text(i, v + TOTAL_CONCEPTS * 0.003, f'{v:,}', ha='center', fontsize=10)
ax.set_ylabel('concepts')
ax.set_title('Gold-ID validity')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Candidate pool statistics

In [ ]:
print('Candidate pool size per concept:')
print(df['n_candidates'].describe().to_string())
print(f"\nExpected 10 candidates: {(df['n_candidates'] == 10).mean()*100:.1f}% of concepts")
print(f"Concepts with < 10 candidates: {(df['n_candidates'] < 10).sum():,}")

# Distinct IDs across all candidate pools
all_cand_ids = [cid for cids in df['candidate_ids'] for cid in cids]
n_distinct_cands = len(set(all_cand_ids))
print(f"\nTotal candidate slots     : {len(all_cand_ids):,}")
print(f"Distinct pictogram IDs    : {n_distinct_cands:,}")
print(f"Mean reuse per pictogram  : {len(all_cand_ids)/n_distinct_cands:.1f}x")

# Top-20 most repeated candidate IDs
cand_freq = Counter(all_cand_ids)
top_cands = pd.DataFrame(cand_freq.most_common(20), columns=['pictogram_id', 'n_appearances'])
print("\nTop-20 most frequent candidate IDs:")
display(top_cands)

## 6. Concept-text quality

In [ ]:
_STOPWORDS = set(stopwords.words('english'))

def _concept_stats(text: str) -> dict:
    tokens = re.findall(r'[a-z]+', str(text).lower())
    n_tok  = len(tokens)
    n_sw   = sum(1 for t in tokens if t in _STOPWORDS)
    return {
        'n_chars':    len(str(text)),
        'n_tokens':   n_tok,
        'sw_ratio':   n_sw / n_tok if n_tok else 0,
        'is_degenerate': (
            n_tok == 0
            or len(str(text).strip()) <= 1
            or str(text).strip().isdigit()
            or (n_tok > 0 and all(t in _STOPWORDS for t in tokens))
        ),
    }

stat_df = df['concept_text'].apply(_concept_stats).apply(pd.Series)
df = pd.concat([df, stat_df], axis=1)

print('Concept-text token length:')
print(df['n_tokens'].describe().to_string())
print(f"\nDegenerate concepts (empty/numeric/stop-word-only): {df['is_degenerate'].sum():,}")
if df['is_degenerate'].any():
    display(df[df['is_degenerate']][['sentence', 'concept_text']].head(15))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(df['n_tokens'], bins=range(0, df['n_tokens'].max() + 2),
        color='#4C9BE8', edgecolor='white')
ax.set_xlabel('tokens in concept_text')
ax.set_ylabel('concepts')
ax.set_title('Concept length (tokens)')
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
ax.hist(df['sw_ratio'], bins=20, color='#9B4CE8', edgecolor='white')
ax.set_xlabel('stop-word ratio')
ax.set_ylabel('concepts')
ax.set_title('Stop-word ratio in concept_text')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 7. Gold-ID distribution

In [ ]:
gold_counts = df_a['gold_id'].value_counts()
n_unique_gold = gold_counts.nunique()

print(f'Unique gold IDs selected: {len(gold_counts):,}')
print(f'Gold IDs used only once : {(gold_counts == 1).sum():,}  ({(gold_counts == 1).mean()*100:.1f}%)')
print(f'Gold IDs used ≥10 times : {(gold_counts >= 10).sum():,}')
print('\nTop-20 most selected gold IDs:')
display(gold_counts.head(20).rename('n_selections').reset_index().rename(columns={'index': 'gold_id'}))

# Concepts per sentence distribution
print('\nConcepts per sentence:')
print(df.groupby('sent_idx')['turn_pos'].count().describe().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Gold ID frequency (log scale)
ax = axes[0]
ax.hist(gold_counts.values, bins=50, color='#4CE8A4', edgecolor='white', log=True)
ax.set_xlabel('times a gold ID is selected')
ax.set_ylabel('number of pictogram IDs (log)')
ax.set_title('Gold ID reuse distribution')
ax.spines[['top', 'right']].set_visible(False)

# Concepts per sentence
ax = axes[1]
n_conc = df.groupby('sent_idx')['turn_pos'].count()
ax.hist(n_conc, bins=range(1, n_conc.max() + 2), color='#E8C84C', edgecolor='white')
ax.set_xlabel('concepts per sentence')
ax.set_ylabel('sentences')
ax.set_title('Concepts-per-sentence distribution')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 8. Gold-ID rank within candidates

Where does the selected gold ID sit within its candidate list?  Position 0 =
first candidate (trivial pick); a uniform distribution is ideal.

In [ ]:
# Fix C: use list() to convert ndarray before calling .index()
def _gold_rank(row):
    if pd.isna(row['gold_id']) or not row['gold_in_candidates']:
        return np.nan
    try:
        return list(row['candidate_ids']).index(int(row['gold_id']))
    except ValueError:
        return np.nan

df_a = df[df['gold_id'].notna()].copy()
df_a['gold_rank'] = df_a.apply(_gold_rank, axis=1)

rank_counts = df_a['gold_rank'].dropna().astype(int).value_counts().sort_index()
print('Gold ID rank within candidate list (0 = first candidate):')
print(rank_counts.to_string())
print(f"\nMean rank   : {df_a['gold_rank'].mean():.2f}  (ideal ≈ 4.5 for 10 candidates)")
print(f"Median rank : {df_a['gold_rank'].median():.1f}")

# Bias check: if rank 0 is heavily over-represented the model always picks first candidate
pct_rank0 = (rank_counts.get(0, 0) / rank_counts.sum()) * 100
print(f"Rank-0 selections: {pct_rank0:.1f}%  {'⚠ possible first-pick bias' if pct_rank0 > 30 else '✓ OK'}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rank_counts.index, rank_counts.values, color='#4C9BE8', edgecolor='white')
# Reference uniform line
ax.axhline(rank_counts.sum() / 10, color='red', linestyle='--', linewidth=1, label='uniform baseline')
ax.set_xlabel('rank of gold ID in candidate list')
ax.set_ylabel('count')
ax.set_title('Position of selected gold ID within 10 candidates')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 9. Caregiver text quality

Analyses the LLM-generated `caregiver_clear` and `caregiver_vague` columns:
word-length distributions, relative verbosity (clear vs vague), residual `{TIME}`
placeholders, and null/empty fill rate.

In [ ]:
# ── Helper: word count (handles NaN gracefully) ──────────────────────────
def _wc(text) -> int:
    if pd.isna(text) or str(text).strip() == '':
        return 0
    return len(str(text).split())

df_ann['_wc_clear'] = df_ann['caregiver_clear'].apply(_wc)
df_ann['_wc_vague'] = df_ann['caregiver_vague'].apply(_wc)

# ── Null / empty counts ──────────────────────────────────────────────────
n_null_clear = (df_ann['caregiver_clear'].isna() | (df_ann['caregiver_clear'].astype(str).str.strip() == '')).sum()
n_null_vague = (df_ann['caregiver_vague'].isna() | (df_ann['caregiver_vague'].astype(str).str.strip() == '')).sum()
print(f'Rows with empty caregiver_clear : {n_null_clear:,}  ({n_null_clear/len(df_ann)*100:.1f}%)')
print(f'Rows with empty caregiver_vague : {n_null_vague:,}  ({n_null_vague/len(df_ann)*100:.1f}%)')

# ── Residual {TIME} placeholders (should be 0 after render) ─────────────
n_time_clear = df_ann['caregiver_clear'].astype(str).str.contains(r'\{TIME\}', regex=True).sum()
n_time_vague = df_ann['caregiver_vague'].astype(str).str.contains(r'\{TIME\}', regex=True).sum()
print(f'Residual {{TIME}} in caregiver_clear : {n_time_clear:,}  (expected 0 after render)')
print(f'Residual {{TIME}} in caregiver_vague : {n_time_vague:,}  (expected 0 after render)')

# ── Verbosity: vague should be shorter than clear ────────────────────────
mean_clear = df_ann['_wc_clear'].mean()
mean_vague = df_ann['_wc_vague'].mean()
print(f'\nMean word count — caregiver_clear : {mean_clear:.1f}')
print(f'Mean word count — caregiver_vague : {mean_vague:.1f}')
pct_vague_shorter = (df_ann['_wc_vague'] < df_ann['_wc_clear']).mean() * 100
print(f'Rows where len(vague) < len(clear): {pct_vague_shorter:.1f}%  (expected >> 50%)')

# ── Word-length histograms ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
max_wc = max(df_ann['_wc_clear'].max(), df_ann['_wc_vague'].max(), 1)
bins = range(0, int(max_wc) + 2)
ax.hist(df_ann['_wc_clear'], bins=bins, alpha=0.7, color='#4C9BE8', label='caregiver_clear')
ax.hist(df_ann['_wc_vague'], bins=bins, alpha=0.7, color='#F28B30', label='caregiver_vague')
ax.set_xlabel('word count')
ax.set_ylabel('sentences')
ax.set_title('Caregiver text length distribution')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
diff = df_ann['_wc_clear'] - df_ann['_wc_vague']
ax.hist(diff, bins=range(int(diff.min()) - 1, int(diff.max()) + 2),
        color='#4CE8A4', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('word count difference (clear − vague)')
ax.set_ylabel('sentences')
ax.set_title('clear − vague word count (positive = clear is longer)')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 10. Time-of-day distribution

Pie chart of `time_of_day` slot counts and `tod_selection` method
(model-forced / keyword-forced / sampled).

In [ ]:
if 'time_of_day' not in df_ann.columns:
    print('time_of_day column not found — skipping section 10.')
else:
    tod_counts = df_ann['time_of_day'].value_counts()
    print('time_of_day distribution:')
    print(tod_counts.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    colors = ['#FFD166', '#06D6A0', '#118AB2', '#073B4C']
    ax.pie(
        tod_counts.values,
        labels=tod_counts.index,
        autopct='%1.1f%%',
        colors=colors[:len(tod_counts)],
        startangle=90,
    )
    ax.set_title('time_of_day distribution')

    ax = axes[1]
    if 'tod_selection' in df_ann.columns:
        sel_counts = df_ann['tod_selection'].fillna('unknown').value_counts()
        ax.bar(sel_counts.index, sel_counts.values, color='#9B4CE8', edgecolor='white')
        for i, v in enumerate(sel_counts.values):
            ax.text(i, v + len(df_ann) * 0.003, f'{v:,}', ha='center', fontsize=9)
        ax.set_xlabel('tod_selection method')
        ax.set_ylabel('sentences')
        ax.set_title('tod_selection: how time_of_day was assigned')
        ax.spines[['top', 'right']].set_visible(False)
        print('\ntod_selection distribution:')
        print(sel_counts.to_string())
    else:
        ax.text(0.5, 0.5, 'tod_selection column not present', ha='center', va='center',
                transform=ax.transAxes)
        ax.set_title('tod_selection (not available)')

    plt.tight_layout()
    plt.show()

## 11. Event-time distribution

Histogram of `event_time` (HH:MM) across 24 hours.
Should roughly follow the band weights defined in the annotation notebook
(morning 30%, afternoon 30%, evening 20%, night 20%) with no artificial spikes.

In [ ]:
if 'event_time' not in df_ann.columns:
    print('event_time column not found — skipping section 11.')
else:
    def _to_decimal_hour(t):
        """Parse HH:MM to decimal hour, returns None on failure."""
        if pd.isna(t) or str(t).strip() == '':
            return None
        try:
            h, m = map(int, str(t).split(':'))
            return h + m / 60
        except Exception:
            return None

    hours = df_ann['event_time'].apply(_to_decimal_hour).dropna()
    n_missing = len(df_ann) - len(hours)
    print(f'Parsed event_time    : {len(hours):,}')
    print(f'Missing / unparseable: {n_missing:,}')
    print(f'Min: {hours.min():.2f}h   Max: {hours.max():.2f}h   Mean: {hours.mean():.2f}h')

    hour_buckets = hours.apply(lambda h: int(h)).value_counts().sort_index()
    print('\nCounts per hour (0–23):')
    print(hour_buckets.to_string())

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.bar(hour_buckets.index, hour_buckets.values, width=0.8,
           color='#4C9BE8', edgecolor='white')
    for x, label in [(5, 'morning'), (13, 'afternoon'), (18, 'evening'), (21, 'night')]:
        ax.axvline(x, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
        ax.text(x + 0.15, ax.get_ylim()[1] * 0.92, label, fontsize=7, color='red', alpha=0.8)
    ax.set_xlabel('hour of day')
    ax.set_ylabel('sentences')
    ax.set_title('event_time distribution across 24 hours')
    ax.set_xticks(range(0, 24))
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

## 12. Schedule quality

Analyses the `schedule` column (list of calendar events):
fill rate, number of events per row, and field-level completeness
(`title`, `start_time`, `location`, `description`).

In [ ]:
if 'schedule' not in df_ann.columns:
    print('schedule column not found — skipping section 12.')
else:
    import ast as _ast

    def _parse_sched(v):
        if isinstance(v, list):
            return v
        if pd.isna(v):
            return []
        try:
            return _ast.literal_eval(str(v))
        except Exception:
            return []

    schedules = df_ann['schedule'].apply(_parse_sched)
    n_events_per_row = schedules.apply(len)

    n_empty = (n_events_per_row == 0).sum()
    print(f'Rows with empty schedule (no events) : {n_empty:,}  ({n_empty/len(df_ann)*100:.1f}%)')
    print(f'Rows with >= 1 event                 : {(n_events_per_row >= 1).sum():,}')
    print(f'\nEvent count per row:')
    print(n_events_per_row.value_counts().sort_index().to_string())
    print(f'Mean events per row: {n_events_per_row.mean():.2f}')

    all_events = [ev for sched in schedules for ev in sched if isinstance(ev, dict)]
    n_total_events = len(all_events)
    if n_total_events > 0:
        print('\nField fill rate across all events:')
        for field in ('title', 'start_time', 'location', 'description'):
            n_filled = sum(1 for ev in all_events
                          if ev.get(field) is not None and str(ev.get(field, '')).strip() != '')
            print(f'  {field:<15}: {n_filled:,}/{n_total_events:,}  ({n_filled/n_total_events*100:.1f}%)')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    vc = n_events_per_row.value_counts().sort_index()
    ax.bar(vc.index.astype(str), vc.values, color='#E8C84C', edgecolor='white')
    ax.set_xlabel('events in schedule')
    ax.set_ylabel('sentences')
    ax.set_title('Number of schedule events per sentence')
    ax.spines[['top', 'right']].set_visible(False)

    ax = axes[1]
    if n_total_events > 0:
        fields = ('title', 'start_time', 'location', 'description')
        fill_rates = [
            sum(1 for ev in all_events
                if ev.get(f) is not None and str(ev.get(f, '')).strip() != '') / n_total_events * 100
            for f in fields
        ]
        ax.bar(fields, fill_rates, color='#4CE8A4', edgecolor='white')
        ax.set_ylim(0, 110)
        ax.set_ylabel('fill rate (%)')
        ax.set_title('Schedule event field fill rate')
        ax.spines[['top', 'right']].set_visible(False)
    else:
        ax.text(0.5, 0.5, 'No events found', ha='center', va='center', transform=ax.transAxes)

    plt.tight_layout()
    plt.show()

## 13. Distractor events

A distractor is a schedule event whose `start_time` differs from the row’s
`event_time` by ≥ 4 hours (i.e., it occurs in a different time band and acts
as a temporal foil for the AAC agent).  Checks whether the current pipeline
produces distractors, and if so analyses their distribution.

In [ ]:
if 'schedule' not in df_ann.columns or 'event_time' not in df_ann.columns:
    print('schedule or event_time column not found — skipping section 13.')
else:
    import ast as _ast

    def _parse_sched_d(v):
        if isinstance(v, list):
            return v
        if pd.isna(v):
            return []
        try:
            return _ast.literal_eval(str(v))
        except Exception:
            return []

    def _to_minutes(t):
        """Parse HH:MM to total minutes from midnight."""
        if pd.isna(t) or str(t).strip() == '':
            return None
        try:
            h, m = map(int, str(t).split(':'))
            return h * 60 + m
        except Exception:
            return None

    DISTRACTOR_GAP_MINUTES = 4 * 60  # >= 4 hours

    distractor_titles = []
    n_rows_with_distractor = 0

    for _, row in df_ann.iterrows():
        event_min = _to_minutes(row.get('event_time'))
        sched = _parse_sched_d(row.get('schedule'))
        row_has_distractor = False
        for ev in sched:
            if not isinstance(ev, dict):
                continue
            ev_min = _to_minutes(ev.get('start_time'))
            if event_min is not None and ev_min is not None:
                gap = abs(ev_min - event_min)
                gap = min(gap, 1440 - gap)  # wrap around midnight
                if gap >= DISTRACTOR_GAP_MINUTES:
                    distractor_titles.append(ev.get('title', '<no title>'))
                    row_has_distractor = True
        if row_has_distractor:
            n_rows_with_distractor += 1

    print(f'Rows with >= 1 distractor event (gap >= 4h): {n_rows_with_distractor:,}')
    print(f'  ({n_rows_with_distractor/len(df_ann)*100:.1f}% of all sentences)')
    print(f'Total distractor events found: {len(distractor_titles):,}')

    if distractor_titles:
        title_counts = pd.Series(distractor_titles).value_counts().head(20)
        print('\nTop-20 distractor titles:')
        print(title_counts.to_string())

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.barh(title_counts.index[::-1], title_counts.values[::-1],
                color='#E84C4C', edgecolor='white')
        ax.set_xlabel('occurrences')
        ax.set_title('Top distractor event titles (gap >= 4h from event_time)')
        ax.spines[['top', 'right']].set_visible(False)
        plt.tight_layout()
        plt.show()
    else:
        print('\nNo distractors found — the current pipeline does not produce temporal foils.')
        print('This is expected for the Qwen annotation run (distractors were a Mistral post-processing step).')

## 9. Annotation log analysis

The cluster writes a JSONL log with one entry per annotated row.
Analyses throughput, error rates, and batch-level timing.

In [ ]:
if not LOG_PATH.exists():
    print(f'Log not found at {LOG_PATH} — skipping log analysis.')
else:
    log_records = []
    with open(LOG_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    log_records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

    df_log = pd.DataFrame(log_records)
    print(f'Log entries : {len(df_log):,}')
    print(f'Columns     : {list(df_log.columns)}')
    display(df_log.head(5))

In [ ]:
if 'df_log' in dir() and len(df_log) > 1:
    # Try common timestamp column names
    ts_col = next((c for c in ['timestamp', 'ts', 'time', 'created_at'] if c in df_log.columns), None)

    if ts_col:
        df_log['_ts'] = pd.to_datetime(df_log[ts_col], errors='coerce')
        df_log = df_log.sort_values('_ts').reset_index(drop=True)
        df_log['_delta_s'] = df_log['_ts'].diff().dt.total_seconds()

        elapsed = (df_log['_ts'].iloc[-1] - df_log['_ts'].iloc[0]).total_seconds()
        throughput = len(df_log) / elapsed * 3600  # rows/hour

        print(f'Total wall time : {elapsed/3600:.2f} h')
        print(f'Throughput      : {throughput:.0f} rows/hour')
        print(f'Median row time : {df_log["_delta_s"].median():.2f} s')
        print(f'p95 row time    : {df_log["_delta_s"].quantile(0.95):.2f} s')

        fig, ax = plt.subplots(figsize=(11, 3))
        ax.plot(df_log.index, df_log['_delta_s'].rolling(50).mean(),
                color='#4C9BE8', linewidth=1)
        ax.set_xlabel('log entry index')
        ax.set_ylabel('seconds (50-row MA)')
        ax.set_title('Annotation throughput over time')
        ax.spines[['top', 'right']].set_visible(False)
        plt.tight_layout()
        plt.show()
    else:
        print('No timestamp column found in log — skipping timing analysis.')

    # Error/skip rate
    if 'status' in df_log.columns:
        print('\nStatus distribution:')
        print(df_log['status'].value_counts().to_string())
    elif 'error' in df_log.columns:
        n_err = df_log['error'].notna().sum()
        print(f'\nRows with error field set: {n_err:,}  ({n_err/len(df_log)*100:.1f}%)')

## 10. Sentences with suspect annotation

Flags sentences where the annotation looks potentially wrong:
- All gold IDs in the sentence are the same (model might have been lazy)
- Gold ID is always rank 0 (model always picks the first candidate)
- Concept text and the gold-ID selection diverge strongly (short/degenerate text but specific ID)

In [ ]:
# Flag 1: all gold_ids identical within a sentence
sent_gold_unique = (
    df_a.groupby('sent_idx')['gold_id']
    .nunique()
    .rename('unique_golds')
)
df_ann_meta = df_ann.copy()
df_ann_meta['n_concepts'] = df_ann_meta['concepts'].apply(len)
df_ann_meta = df_ann_meta.join(sent_gold_unique)

# Only meaningful for sentences with ≥2 concepts
suspect_same = df_ann_meta[
    (df_ann_meta['n_concepts'] >= 2) &
    (df_ann_meta['unique_golds'] == 1)
]
print(f'Sentences with ≥2 concepts but all same gold_id: {len(suspect_same):,}')
if len(suspect_same):
    display(suspect_same[['sentence', 'n_concepts', 'unique_golds']].head(10))

# Flag 2: gold always at rank 0 (in sentences with ≥2 concepts)
if 'gold_rank' in df_a.columns:
    always_rank0 = (
        df_a[df_a['n_concepts'] >= 2]
        .groupby('sent_idx')['gold_rank']
        .apply(lambda s: (s == 0).all())
    )
    n_always0 = always_rank0.sum()
    print(f'\nSentences (≥2 concepts) where gold is ALWAYS rank-0: {n_always0:,}')

## 11. Random sample inspection

In [ ]:
SAMPLE_N = 10
SEED     = 42
random.seed(SEED)

sample_idx = random.sample(list(df_ann.index), min(SAMPLE_N, len(df_ann)))

for i, idx in enumerate(sample_idx):
    row = df_ann.iloc[idx]
    print(f'\n──── Sample {i+1} (row {idx}) ──────────────────────────────────────')
    print(f'sentence : {row["sentence"]}')
    for c in row['concepts']:
        gid  = c.get('gold_id')
        cids = c.get('candidate_ids', [])
        # Fix C: convert ndarray to list before calling .index()
        cids_list = list(cids)
        rank = cids_list.index(int(gid)) if gid is not None and int(gid) in cids_list else '∉ candidates'
        valid = '✓' if gid in cids else '✗'
        print(f'  [{valid} rank={rank}]  concept={c.get("concept_text")!r:35s}  gold={gid}')

## 12. Summary

In [ ]:
print('=' * 60)
print('  ANNOTATION QUALITY SUMMARY')
print('=' * 60)
print(f'  Sentences          : {TOTAL_SENTENCES:,}')
print(f'  Concepts           : {TOTAL_CONCEPTS:,}')
print(f'  Completeness       : {n_annotated/TOTAL_CONCEPTS*100:.1f}%')
print(f'  Gold-in-candidates : {n_in/len(df_a)*100:.1f}%')
print(f'  Degenerate concepts: {df["is_degenerate"].sum():,}  ({df["is_degenerate"].mean()*100:.1f}%)')
if 'gold_rank' in df_a.columns:
    print(f'  Mean gold rank     : {df_a["gold_rank"].mean():.2f}  (ideal ≈ 4.5)')
    print(f'  Rank-0 bias        : {pct_rank0:.1f}%')
print(f'  Suspect sentences  : {len(suspect_same):,}  (same gold for all concepts)')
print('=' * 60)